In [ ]:
# Distinct Imports
!pip install -q torchinfo accelerate tqdm

import os
import pandas as pd 
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch import Tensor
from accelerate import Accelerator
from torchinfo import summary
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from tqdm import tqdm
from PIL import Image
from collections import defaultdict
from IPython.display import clear_output
from tqdm import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time
import cv2
import torch.autograd as autograd
import math
import copy
from sklearn.linear_model import Lasso

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

: 

In [ ]:
class SparseSignalsDataset(Dataset):
    def __init__(self, file_path,samples_number=10):
        data = np.load(file_path)[:samples_number]
        data = data.T
        self.data = torch.tensor(data).float()
        self.n_samples = self.data.shape[1]
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, index):
        return self.data[:, index]

def collate_signals(batch):
    return torch.stack(batch, dim=1)



train_dataset_file_path = '/kaggle/input/sparsesignal/sparse_signals_dataset_10000.npy'
test_dataset_file_path = '/kaggle/input/sparsesignal/sparse_signals_dataset_1000.npy'
train_dataset = SparseSignalsDataset(train_dataset_file_path,samples_number=4500)  
test_dataset = SparseSignalsDataset(test_dataset_file_path,samples_number=1000)   

batch_size = 64
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_signals)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=2 , collate_fn=collate_signals)

n = 100  #Signal length
'-----------Gaussian Sensing Matrix Dimension----------'
m_ = 15
Gaussian_A = torch.normal(0, 1, (m_, n)).to(device)
'-----------Learned Sensing Matrix Dimension-----------'
m = 15
'------------------------------------------------------'

In [ ]:
def L1Lasso(A, Y, alpha=2.812e-01, tol=1e-4):
    A_np = A.detach().cpu().numpy()
    Y_np = Y.detach().cpu().numpy()
    lasso = Lasso(alpha=alpha, fit_intercept=False, max_iter=50000, tol=tol)
    lasso.fit(A_np, Y_np)
    X_np = lasso.coef_
    X_np = X_np.T  
    X = torch.tensor(X_np).to(A.device)
    return X

In [ ]:
def sup_xxhat(x_true, x_hat, device='cpu'):
    non_zero_mask_true = x_true != 0 
    non_zero_mask_hat = x_hat != 0  
    top_15_indices_true = torch.argsort(torch.abs(x_true), dim=0, descending=True)[:10, :]
    top_15_indices_hat = torch.argsort(torch.abs(x_hat), dim=0, descending=True)[:10, :]
    top_15_mask_true = torch.zeros_like(x_true, dtype=torch.bool, device=device)
    top_15_mask_hat = torch.zeros_like(x_hat, dtype=torch.bool, device=device)
    top_15_mask_true.scatter_(0, top_15_indices_true, True)
    top_15_mask_hat.scatter_(0, top_15_indices_hat, True)
    support_true = (non_zero_mask_true & top_15_mask_true) 
    support_hat = (non_zero_mask_hat & top_15_mask_hat) 
    intersection = (support_true & support_hat).sum(dim=0).float() 
    union = (support_true ).sum(dim=0).float()
    support_ratios = intersection / (union)
    sum_support_ratio = support_ratios.sum().item()
    return sum_support_ratio

def nmse_xxhat(x_true, x_hat, device='cpu'):
    numerator = torch.sum((x_true - x_hat)**2 , dim=0) 
    denominator = torch.sum(x_true**2 , dim=0)          
    nmse_per_sample = numerator / (denominator) 
    sum_nmse = torch.sum(nmse_per_sample).item()
    return sum_nmse
    
def signal_recovery_eval(x_hat, x_true, device):
    sum_nmse = nmse_xxhat(x_true = x_true , x_hat = x_hat,device=device)
    sum_support = sup_xxhat(x_true = x_true,x_hat=x_hat,device=device)
    return sum_nmse , sum_support

In [ ]:
def conjugate_gradient(H, b, tol=1e-6, max_iter=100): #used to solve Hx = b, where H is function 
                                                      #that calculates Hx.
    x = torch.zeros_like(b)
    r = b.clone()
    p = r.clone()
    rsold = torch.sum(r*r)

    for i in range(max_iter):
        Hp = H(p)
        alpha = rsold / torch.sum(p* Hp)
        x = x + alpha * p
        r = r - alpha * Hp
        rsnew = torch.sum(r*r)

        if torch.sqrt(rsnew) < tol:
            break

        p = r + (rsnew / rsold) * p
        rsold = rsnew

    return x

def compute_Hv(loss, p, u, v, flag="both"): 
    grad_u = autograd.grad(loss, u, create_graph=True)[0]
    
    if flag == "grad":
        return grad_u
    
    elif flag == "both":
        dell_u_times_p = torch.sum(grad_u * p)
        Hv = autograd.grad(dell_u_times_p, v, retain_graph=True)[0]
        return grad_u, Hv
    
    elif flag == "hess":
        dell_u_times_p = torch.sum(grad_u * p)
        Hv = autograd.grad(dell_u_times_p, v, retain_graph=True)[0]
        return Hv

def create_H_function(loss, u, v):
    def H(p):
        return compute_Hv(loss, p, u, v, flag="hess")
    return H

In [ ]:
def outer_loss_fn(pred_x, x_true):
    criterion = nn.MSELoss(reduction='none')
    mse_loss_per_feature = criterion(pred_x, x_true)  # Shape: [n, N]
    mse_loss_per_sample = mse_loss_per_feature.sum(dim=0)  # Shape: [1,N]
    total_loss = mse_loss_per_sample.mean()    
    return total_loss

In [ ]:
class SmoothL1Regularizer(nn.Module):
    def __init__(self, n, init_sigma=-7.0, init_lambda0=0.0, device='cpu'):
        super().__init__()
        self.sigma = nn.Parameter(torch.tensor(init_sigma, dtype=torch.float32, device=device))
        self.lambda0 = nn.Parameter(torch.tensor(init_lambda0, dtype=torch.float32, device=device))
        self.W = nn.Parameter(torch.eye(n, dtype=torch.float32, device=device))
        # self.sigma.data.clamp_(min=-10, max=0)  # Constrain sigma to reasonable range

    def forward(self, x_hat):
        """Computes the SmoothL1 regularization term"""
        transform_xhat = self.W @ x_hat
        regularization = torch.sqrt(transform_xhat**2 + torch.exp(self.sigma))
        return torch.exp(self.lambda0) * torch.sum(regularization, dim=0)

def inner_loss(Y, A, x_hat, regularizer):
    """Computes the total loss with integrated regularizer"""
    residual = Y - A @ x_hat
    loss1 = torch.sum(residual**2, dim=0)
    regularizer_term = regularizer(x_hat)
    return (loss1 + regularizer_term).mean()

def inner_optimization(x_hat, Y, A, regularizer, tol=1e-9, max_iter=500):
    recons_x = x_hat.requires_grad_(True)
    optimizer = optim.LBFGS([recons_x], 
                      lr=0.1,
                      line_search_fn='strong_wolfe',  
                      tolerance_grad=1e-8,  
                      tolerance_change=1e-10)

    def closure():
        optimizer.zero_grad()
        loss = inner_loss(Y=Y, A=A, x_hat=recons_x, regularizer=regularizer)
        loss.backward(retain_graph=True)
        return loss
    
    optimizer.step(closure)
    return recons_x

In [ ]:
def hoag_algorithm(x_hat, x_true, A, regularizer, outer_loss_fn, device, max_iter=20, k=1.0, epsilon=1e-8):
    tol = 1e-3
    Y = (A @ x_true).to(device)
    
    params = [A] + list(regularizer.parameters())
    gradients = []

    recons_x = inner_optimization(x_hat=x_hat, Y=Y, A=A, regularizer=regularizer, tol=tol)
    outer_loss = outer_loss_fn(recons_x, x_true)
    grad_g_x = autograd.grad(outer_loss, recons_x, create_graph=True)[0]
    
    p1 = torch.linalg.matrix_norm(grad_g_x).mean().item()
    L = k * p1
    
    for iteration in range(max_iter):
        tol = min(tol * (1 - iteration / max_iter)**2, epsilon)  
        recons_x = inner_optimization(x_hat=x_hat, Y=Y, A=A, regularizer=regularizer, tol=tol)
        x_hat = recons_x.detach().requires_grad_(True)
        new_inner_loss = inner_loss(Y=Y, A=A, x_hat=x_hat, regularizer=regularizer)
        Hxx = create_H_function(new_inner_loss, x_hat, x_hat)

        current_loss = outer_loss.item()
        cg_tol = max(1e-6, min(1e-3, current_loss * 1e-3)) 
        q_k = conjugate_gradient(Hxx, grad_g_x, 
                                         tol=cg_tol, 
                                         max_iter=50)
            
        gradients.clear()
        for param in params:
            Hv = compute_Hv(new_inner_loss, q_k, x_hat, param, flag="hess")
            gradients.append(-Hv)
    
    return [gradients[0]] + gradients[1:] + [L, outer_loss.item(), x_hat]


In [ ]:
A = nn.Parameter(torch.randn(m, n, dtype=torch.float32, device=device))
regularizer = SmoothL1Regularizer(n=n, device=device).to(device)

print(f'Initial A (device: {A.device}):\n{A}')
print(f'Regularizer sigma (device: {regularizer.sigma.device}): {regularizer.sigma}')
print(f'Regularizer lambda0 (device: {regularizer.lambda0.device}): {regularizer.lambda0}')
print(f'Regularizer W (device: {regularizer.W.device}):\n{regularizer.W}')

In [ ]:
print(f"Training Dataset Size: {len(train_dataset)}")
print(f"Testing Dataset Size: {len(test_dataset)}")

train_nmse_loss_history = []
test_nmse_loss_history = []
train_support_loss_history = []
test_support_loss_history = []

optimizer = optim.Adam([
    {'params': [A], 'lr': 0.1},
    {'params': regularizer.parameters(), 'lr': 0.001}
], weight_decay=1e-2)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)


previous_estimates = []  
max_epochs = 100
total_tr_time = 0
for epoch in range(max_epochs):
    epoch_start_time = time.time()
    
    running_nmse_loss = 0.0
    total_train_samples = 0
    total_nmse_loss = 0.0
    total_support_loss = 0.0
    
    train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch + 1} Training", leave=False)
    for batch_idx, x_true in enumerate(train_loader_tqdm):
        n_samples = x_true.shape[1]
        total_train_samples += n_samples
        x_true = x_true.to(device)
        
        # Initialize previous_estimates list if necessary
        if len(previous_estimates) < math.ceil(len(train_dataset) / batch_size):
            previous_estimates.append(torch.zeros_like(x_true).to(device))
            
        initial_estimate = previous_estimates[batch_idx]
        
        # Updated training loop usage
        grad_A, grad_sigma, grad_lambda0, grad_W, L , loss, x_hat = hoag_algorithm(
            x_hat=initial_estimate,
            x_true=x_true,
            A=A,
            regularizer=regularizer,
            outer_loss_fn=outer_loss_fn,
            device=device,
            max_iter=20,
            k=1.0,
            epsilon=1e-5
        )
        
        # Update stored estimate
        previous_estimates[batch_idx] = x_hat

        # Update parameters
        optimizer.zero_grad()
        A.grad = grad_A
        regularizer.sigma.grad = grad_sigma
        regularizer.lambda0.grad = grad_lambda0
        # regularizer.W.grad = grad_W
        optimizer.step()
        
        # Evaluate performance for this batch
        nmse_loss_batch, support_loss_batch = signal_recovery_eval(
            x_hat=x_hat,
            x_true=x_true,
            device=device
        )
        total_nmse_loss += nmse_loss_batch
        total_support_loss += support_loss_batch
        running_nmse_loss += nmse_loss_batch
        
        train_loader_tqdm.set_postfix({
            'Outer Loss': loss, 
            'Avg NMSE Loss': running_nmse_loss / total_train_samples  
        })
        
    avg_train_nmse_loss = total_nmse_loss / total_train_samples
    avg_train_support_loss = total_support_loss / total_train_samples
    train_nmse_loss_history.append(avg_train_nmse_loss)
    train_support_loss_history.append(avg_train_support_loss)
    
    # Adjust learning rate based on training loss
    scheduler.step(avg_train_nmse_loss)

    total_tr_time+=time.time() - epoch_start_time
    # Testing phase
    with torch.no_grad():
        total_test_samples = 0
        total_nmse_test_loss = 0.0
        total_support_test_loss = 0.0
        
        test_loader_tqdm = tqdm(test_loader, desc=f"Epoch {epoch + 1} Testing", leave=False)
        for batch_idx, x_true in enumerate(test_loader_tqdm):
            n_samples = x_true.shape[1]
            total_test_samples += n_samples
            
            x_true = x_true.to(device)
            initial_estimate = torch.zeros_like(x_true).to(device)

            Y = (A @ x_true).to(device)
            
            x_hat = inner_optimization(x_hat=initial_estimate, Y=Y, A=A, regularizer=regularizer)
            
            if x_hat.dim() == 1:
                x_hat = x_hat.unsqueeze(1)
                
            nmse_loss_batch, support_loss_batch = signal_recovery_eval(
                x_hat=x_hat,
                x_true=x_true,
                device=device
            )
            total_nmse_test_loss += nmse_loss_batch
            total_support_test_loss += support_loss_batch
        
        avg_test_nmse_loss = total_nmse_test_loss / total_test_samples
        avg_test_support_loss = total_support_test_loss / total_test_samples
        test_nmse_loss_history.append(avg_test_nmse_loss)
        test_support_loss_history.append(avg_test_support_loss)


    epoch_elapsed_time = time.time() - epoch_start_time
    
    # Clear, formatted printing for each epoch summary
    tqdm.write(f"\nEpoch {epoch + 1} Summary:")
    tqdm.write(f"  Training Time       : {epoch_elapsed_time:.2f} s")
    tqdm.write(f"  Avg Train NMSE Loss : {avg_train_nmse_loss:.8f}")
    tqdm.write(f"  Avg Train Support   : {avg_train_support_loss:.8f}")
    tqdm.write(f"  Sigma: {regularizer.sigma.item():.4f} | Lambda0: {regularizer.lambda0.item():.4f} | W Grad Norm: {regularizer.W.grad.norm().item():.4f}")
    tqdm.write(f"  Avg Test NMSE Loss  : {avg_test_nmse_loss:.8f}")
    tqdm.write(f"  Avg Test Support    : {avg_test_support_loss:.8f}")

In [ ]:
print(f'Total training time is {total_tr_time} s')

In [ ]:
print(f'Initial A (device: {A.device}):\n{A}')
print(f'Regularizer sigma (device: {regularizer.sigma.device}): {regularizer.sigma}')
print(f'Regularizer lambda0 (device: {regularizer.lambda0.device}): {regularizer.lambda0}')
print(f'Regularizer W (device: {regularizer.W.device}):\n{regularizer.W}')

In [ ]:
def L1Lasso(A, Y, alpha=2.812e-01, tol=1e-4):
    A_np = A.detach().cpu().numpy()
    Y_np = Y.detach().cpu().numpy()
    lasso = Lasso(alpha=alpha, fit_intercept=False, max_iter=50000, tol=tol)
    lasso.fit(A_np, Y_np)
    X_np = lasso.coef_
    X_np = X_np.T  
    X = torch.tensor(X_np).to(A.device)
    return X

def bestAlphaFind(data_loader, gaussian_A, device, num_alphas=50):
    """
    Evaluate reconstruction performance over a range of alpha values and return the best alpha.
    
    Parameters:
        data_loader: DataLoader yielding batches of signals with shape [signal_length, batch_size]
        gaussian_A: Measurement matrix (expects shape [m, signal_length])
        device: torch.device for computations
        num_alphas: Number of candidate alpha values (default: 50)
        
    Returns:
        best_alpha_value: The alpha that resulted in the lowest NMSE.
    """
    alphas = np.logspace(-4, 2, num=num_alphas)
    results = []
    
    for alpha in alphas:
        nmse_list, support_list = [], []
        total_samples = 0
        
        with torch.no_grad():
            for x_trues in tqdm(data_loader, leave=False):
                x_true = x_trues.to(device)
                YG = torch.matmul(gaussian_A, x_true).to(device)
                recons_x_G = L1Lasso(gaussian_A, YG, alpha=alpha)
                if recons_x_G.dim() == 1:
                    recons_x_G = recons_x_G.unsqueeze(1)
                nmse_val, support_val = signal_recovery_eval(recons_x_G, x_true, device=device)
                nmse_list.append(nmse_val)
                support_list.append(support_val)
                total_samples += x_trues.shape[1]
        
        avg_nmse = torch.sum(torch.tensor(nmse_list)) / total_samples
        avg_support = torch.sum(torch.tensor(support_list)) / total_samples
        results.append((alpha, avg_nmse.item(), avg_support.item()))
    best_alpha = min(results, key=lambda x: x[1])
    print("\n|-------- Best Alpha Results --------|")
    print(f"Dataset size: {total_samples}")
    print(f"Best alpha: {best_alpha[0]:.3e}")
    print(f"NMSE: {best_alpha[1]:.8f}")
    print(f"Support Recovery: {best_alpha[2]:.8f}")
    print("|------------------------------------|")
    
    alphas_plot = [r[0] for r in results]
    nmses_plot = [r[1] for r in results]
    
    plt.figure(figsize=(10, 6))
    plt.semilogx(alphas_plot, nmses_plot, 'bo-')
    plt.xlabel("Alpha (log scale)")
    plt.ylabel("NMSE")
    plt.title("NMSE vs. Alpha")
    plt.grid(True)
    plt.show()
    
    best_alpha_value = best_alpha[0]
    return best_alpha_value

best_alpha_value = bestAlphaFind(train_loader,Gaussian_A,device)

In [ ]:
def evaluate_gaussian(loader, Gaussian_A, best_alpha_value, device):
    """
    Evaluate the reconstruction performance of Gaussian_A using the given best_alpha_value.
    
    Parameters:
        loader (DataLoader): Yields batches of signals with shape [signal_length, batch_size]
        Gaussian_A (torch.Tensor): Measurement matrix of shape [m, signal_length]
        best_alpha_value (float): Regularization parameter for L1Lasso
        device (torch.device): Device on which to perform computations
        
    Returns:
        avg_nmse (float): Average NMSE over all samples
        avg_support (float): Average support recovery metric over all samples
    """
    nmse_list, support_list = [], []
    total_samples = 0
    
    with torch.no_grad():
        for x_trues in loader:
            batch_size = x_trues.shape[1]
            total_samples += batch_size
            x_true = x_trues.to(device)
            YG = torch.matmul(Gaussian_A, x_true)
            recons_x = L1Lasso(Gaussian_A, YG, alpha=best_alpha_value)
            if recons_x.dim() == 1:
                recons_x = recons_x.unsqueeze(1)
            nmse_val, support_val = signal_recovery_eval(recons_x, x_true, device=device)
            nmse_list.append(nmse_val)
            support_list.append(support_val)
    
    avg_nmse = torch.sum(torch.tensor(nmse_list)) / total_samples
    avg_support = torch.sum(torch.tensor(support_list)) / total_samples
    return avg_nmse.item(), avg_support.item()


train_avg_nmse_G ,train_avg_support_G = evaluate_gaussian(train_loader,Gaussian_A,best_alpha_value,device)
test_avg_nmse_G, test_avg_support_G = evaluate_gaussian(test_loader,Gaussian_A,best_alpha_value,device)

print(f'##############################################################################################')
print(f'    ----------- Train Dataset Results (Gaussian Sensing Matrix A ({Gaussian_A.shape[0]}x{Gaussian_A.shape[1]})) ---------')
print(f'       Dataset Size will evaluating : {len(train_dataset)}')
print(f'       Average Normalized MSE loss (Gaussian Sensing Matrix) : {train_avg_nmse_G} ')
print(f'       Average Support Recovery (Gaussian Sensing Matrix) )    : {train_avg_support_G} ')

print(f'##############################################################################################')
print(f'    ----------- Test Dataset Results (Gaussian Sensing Matrix A ({Gaussian_A.shape[0]}x{Gaussian_A.shape[1]})) ---------')
print(f'       Dataset Size will evaluating : {len(test_dataset)}')
print(f'       Average Normalized MSE loss (Gaussian Sensing Matrix) ) : {test_avg_nmse_G} ')
print(f'       Average Support Recovery (Gaussian Sensing Matrix) )    : {test_avg_support_G} ')
print(f'##############################################################################################')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(range(1, len(train_nmse_loss_history) + 1), train_nmse_loss_history,
         marker='^', linestyle='-', color='r',
         label=f'Learned Sensing with SmoothedL1 (Training Size: {len(train_dataset)})')
plt.plot(range(1, len(test_nmse_loss_history) + 1), test_nmse_loss_history,
         marker='o', linestyle='-', color='b',
         label=f'Learned Sensing with SmoothedL1 (Testing Size: {len(test_dataset)})')
plt.axhline(y=train_avg_nmse_G, color='g', linestyle='--',
            label=f'Gaussian Sensing with Lasso (Training Size: {len(train_dataset)})')
plt.axhline(y=test_avg_nmse_G, color='y', linestyle='--',
            label=f'Gaussian Sensing with Lasso (Testing Size: {len(test_dataset)})')

plt.xlabel('Epoch')
plt.ylabel('Averaged NMSE Loss')
plt.title(f'Averaged NMSE Loss per Epoch | m = {m}')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(range(1, len(train_support_loss_history) + 1), train_support_loss_history,
         marker='^', linestyle='-', color='r',
         label=f'Learned Sensing with SmoothedL1 (Training Size: {len(train_dataset)})')
plt.plot(range(1, len(test_support_loss_history) + 1), test_support_loss_history,
         marker='o', linestyle='-', color='b',
         label=f'Learned Sensing with SmoothedL1 (Testing Size: {len(test_dataset)})')
plt.axhline(y=train_avg_support_G, color='g', linestyle='--',
            label=f'Gaussian Sensing with Lasso (Training Size: {len(train_dataset)})')
plt.axhline(y=test_avg_support_G, color='y', linestyle='--',
            label=f'Gaussian Sensing with Lasso (Testing Size: {len(test_dataset)})')

plt.xlabel('Epoch')
plt.ylabel('Averaged Support Loss')
plt.title(f'Averaged Support Loss per Epoch | m = {m}')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
def evaluate_reconstruction(data_loader, learned_A, regularizer, Gaussian_A, best_alpha, device):
    learned_nmse_losses = []
    learned_support_losses = []
    gaussian_nmse_losses = []
    gaussian_support_losses = []
    total_num_samples = 0

    for x_true_batch in tqdm(data_loader, desc="Evaluating Reconstruction", leave=False):
        batch_sample_count = x_true_batch.shape[1]
        total_num_samples += batch_sample_count
        x_true = x_true_batch.to(device)
        
        # --- Learned Sensing Reconstruction ---
        Y_learned = torch.matmul(learned_A, x_true).to(device)
        initial_estimate = torch.zeros_like(x_true).to(device)
        reconstructed_x_learned = inner_optimization(
            x_hat=initial_estimate,
            Y=Y_learned,
            A=learned_A,
            regularizer=regularizer,
            tol=1e-5
        ).to(device)
        
        nmse_loss_learned, support_loss_learned = signal_recovery_eval(
            x_hat=reconstructed_x_learned,
            x_true=x_true,
            device=device
        )
        learned_nmse_losses.append(nmse_loss_learned)
        learned_support_losses.append(support_loss_learned)
        
        # --- Gaussian Sensing Reconstruction ---
        Y_gaussian = torch.matmul(Gaussian_A, x_true).to(device)
        reconstructed_x_gaussian = L1Lasso(
            A=Gaussian_A,
            Y=Y_gaussian,
            alpha=best_alpha,
            tol=1e-5
        ).to(device)
        
        if reconstructed_x_gaussian.dim() == 1:
            reconstructed_x_gaussian = reconstructed_x_gaussian.unsqueeze(1)
        
        nmse_loss_gaussian, support_loss_gaussian = signal_recovery_eval(
            x_hat=reconstructed_x_gaussian,
            x_true=x_true,
            device=device
        )
        gaussian_nmse_losses.append(nmse_loss_gaussian)
        gaussian_support_losses.append(support_loss_gaussian)
    
    avg_nmse_learned = torch.sum(torch.tensor(learned_nmse_losses)) / total_num_samples
    avg_support_learned = torch.sum(torch.tensor(learned_support_losses)) / total_num_samples
    avg_nmse_gaussian = torch.sum(torch.tensor(gaussian_nmse_losses)) / total_num_samples
    avg_support_gaussian = torch.sum(torch.tensor(gaussian_support_losses)) / total_num_samples

    results = {
        "learned": {"nmse": avg_nmse_learned.item(), "support": avg_support_learned.item()},
        "gaussian": {"nmse": avg_nmse_gaussian.item(), "support": avg_support_gaussian.item()}
    }
    return results


In [ ]:
train_results = evaluate_reconstruction(train_loader, 
                                      A, 
                                      regularizer,  # Pass regularizer instead of individual params
                                      Gaussian_A, 
                                      best_alpha_value, 
                                      device)

print(f'#############################################################################################')
print(f'    ----------- Train Dataset Results (Learned Sensing Matrix A ({A.shape[0]}x{A.shape[1]})) ----------')
print(f'Dataset Size Evaluated: {len(train_dataset)}')
print(f'Average Normalized MSE Loss (Learned Sensing Matrix) : {train_results["learned"]["nmse"]}')
print(f'Average Support Recovery (Learned Sensing Matrix)    : {train_results["learned"]["support"]}')
print(f'##############################################################################################')
print(f'    ----------- Train Dataset Results (Gaussian Sensing Matrix A ({Gaussian_A.shape[0]}x{Gaussian_A.shape[1]})) ---------')
print(f'Dataset Size Evaluated: {len(train_dataset)}')
print(f'Average Normalized MSE Loss (Gaussian Sensing Matrix): {train_results["gaussian"]["nmse"]}')
print(f'Average Support Recovery (Gaussian Sensing Matrix)   : {train_results["gaussian"]["support"]}')
print(f'##############################################################################################')


In [ ]:
train_results = evaluate_reconstruction(test_loader, 
                                      A, 
                                      regularizer,  # Pass regularizer instead of individual params
                                      Gaussian_A, 
                                      best_alpha_value, 
                                      device)

print(f'#############################################################################################')
print(f'    ----------- Train Dataset Results (Learned Sensing Matrix A ({A.shape[0]}x{A.shape[1]})) ----------')
print(f'Dataset Size Evaluated: {len(train_dataset)}')
print(f'Average Normalized MSE Loss (Learned Sensing Matrix) : {train_results["learned"]["nmse"]}')
print(f'Average Support Recovery (Learned Sensing Matrix)    : {train_results["learned"]["support"]}')
print(f'##############################################################################################')
print(f'    ----------- Train Dataset Results (Gaussian Sensing Matrix A ({Gaussian_A.shape[0]}x{Gaussian_A.shape[1]})) ---------')
print(f'Dataset Size Evaluated: {len(train_dataset)}')
print(f'Average Normalized MSE Loss (Gaussian Sensing Matrix): {train_results["gaussian"]["nmse"]}')
print(f'Average Support Recovery (Gaussian Sensing Matrix)   : {train_results["gaussian"]["support"]}')
print(f'##############################################################################################')

In [ ]:
def reconstruct_sample(true_signal, learned_A, regularizer, Gaussian_A, best_alpha, device, tol=1e-4):
    """
    Reconstructs a signal sample using both the learned sensing method and the Gaussian sensing method.
    
    Parameters:
      true_signal (Tensor): The true signal as a column vector.
      learned_A: Learned sensing matrix.
      regularizer: Instance of SmoothL1Regularizer containing learned parameters.
      Gaussian_A (Tensor): Gaussian sensing matrix.
      best_alpha (float): Best alpha value for L1 Lasso.
      device (torch.device): Computation device.
      tol (float): Tolerance for inner optimization.
      
    Returns:
      Tuple[Tensor, Tensor]: Reconstructed signals (learned, gaussian).
    """
    initial_estimate = torch.zeros_like(true_signal).to(device)
    
    observed_signal_learned = learned_A @ true_signal
    reconstructed_learned = inner_optimization(
        x_hat=initial_estimate,
        Y=observed_signal_learned,
        A=learned_A,
        regularizer=regularizer,
        tol=tol
    ).to(device)
    
    observed_signal_gaussian = torch.matmul(Gaussian_A, true_signal).to(device)
    reconstructed_gaussian = L1Lasso(
        A=Gaussian_A,
        Y=observed_signal_gaussian,
        alpha=best_alpha,
        tol=tol
    ).to(device)
    if reconstructed_gaussian.dim() == 1:
        reconstructed_gaussian = reconstructed_gaussian.unsqueeze(1)
        
    return reconstructed_learned, reconstructed_gaussian


def plot_sample_reconstruction(true_signal, reconstructed_learned, reconstructed_gaussian, x_values, sample_label):
    """
    Plots the true signal alongside the reconstructions.
    
    Parameters:
      true_signal (Tensor): The true signal.
      reconstructed_learned (Tensor): Reconstruction from the learned method.
      reconstructed_gaussian (Tensor): Reconstruction from the Gaussian method.
      x_values (array): x-axis values.
      sample_label (str): Label to denote 'Train' or 'Test' sample.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].plot(x_values, true_signal.cpu().numpy(), label='True Signal', marker='1', color='red')
    axes[0].plot(x_values, reconstructed_learned.cpu().detach().numpy(), 
                 label='Reconstructed (Learned)', marker='2', color='orange')
    axes[0].set_title(f'{sample_label} Sample: True vs Reconstructed (Learned)')
    axes[0].set_xlabel('x values')
    axes[0].set_ylabel('Signal Amplitude')
    axes[0].legend()
    axes[0].grid()
    
    axes[1].plot(x_values, true_signal.cpu().numpy(), label='True Signal', marker='1', color='red')
    axes[1].plot(x_values, reconstructed_gaussian.cpu().detach().numpy(), 
                 label='Reconstructed (Gaussian)', marker='1', color='green')
    axes[1].set_title(f'{sample_label} Sample: True vs Reconstructed (Gaussian)')
    axes[1].set_xlabel('x values')
    axes[1].set_ylabel('Signal Amplitude')
    axes[1].legend()
    axes[1].grid()
    
    plt.tight_layout()
    plt.show()

def plot_difference(true_signal, reconstructed_learned, reconstructed_gaussian, x_values, sample_label):
    diff_learned = true_signal - reconstructed_learned
    diff_gaussian = true_signal - reconstructed_gaussian
    
    plt.figure(figsize=(10, 6))
    plt.plot(x_values, diff_learned.cpu().detach().numpy(), label='Difference (Learned)', marker='x', color='red')
    plt.plot(x_values, diff_gaussian.cpu().detach().numpy(),        label='Difference (Gaussian)', marker='3', color='green')
    plt.title(f'{sample_label} Sample: Difference Plot')
    plt.xlabel('x values')
    plt.ylabel('Difference Amplitude')
    plt.legend()
    plt.grid()
    plt.show()

def process_sample(sample, sample_label, learned_A, regularizer, Gaussian_A, best_alpha, device, tol=1e-4):
    true_signal = sample.to(device).view(-1, 1)
    reconstructed_learned, reconstructed_gaussian = reconstruct_sample(
        true_signal, learned_A, regularizer, Gaussian_A, best_alpha, device, tol=tol
    )
    x_values = np.linspace(1, true_signal.shape[0], true_signal.shape[0])
    plot_sample_reconstruction(true_signal, reconstructed_learned, reconstructed_gaussian, x_values, sample_label)
    plot_difference(true_signal, reconstructed_learned, reconstructed_gaussian, x_values, sample_label)

# Updated usage (assuming regularizer is initialized)
process_sample(train_dataset[0], "Train", A, regularizer, Gaussian_A, best_alpha_value, device)
process_sample(test_dataset[1], "Test", A, regularizer, Gaussian_A, best_alpha_value, device)